# Imports

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager 
from tqdm import tqdm
from bs4 import BeautifulSoup
import time 
import os
from pathlib import Path 

In [2]:
new_bible_languages_url = {
    "english": "https://www.bible.com/bible/12/MAT.1.ASV",              #1
    "masbatenyo": "https://www.bible.com/bible/1222/MAT.1.MSB",         #8
    "tagalog": "https://www.bible.com/bible/2195/MAT.1.ABTAG01",        #11
    "ilonggo": "https://www.bible.com/bible/2190/MAT.1.MBBHIL12",       #14
    "waray": "https://www.bible.com/bible/2198/MAT.1.MBBSAM",           #15
    "bikolano": "https://www.bible.com/bible/890/MAT.1.MBBBIK92",       #16
}

new_bible_books = ["MAT", "MRK", "LUK", "JHN", "ACT", "ROM", "1C0", "2CO", "GAL", "EPH", "PHP", "COL", "1TH", "2TH", "1TI", "2TI", "TIT", "PHM", "HEB", "JAS", "1PE", "2PE", "1JN", "2JN", "3JN", "JUD", "REV"]

In [3]:
print("Setting up WebDriver...")
options = Options()
options.add_argument("--headless")  # run chrome without opening a visual window
options.add_argument("--log-level=3")  # suppress unnecessary logs
options.add_experimental_option('excludeSwitches', ['enable-logging'])

# use WebDriver Manager to handle driver installation/updates automatically
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
print("WebDriver ready.")

# data structures for storing scraped data and statistics
bible_data_new = {}  # {lang: {book: {chapter: [verses]}}}
word_counts_new = {}  # {lang: count}
total_words_new = 0
total_verses_new = 0
total_chapters_new = 0

# progress bar setup
total_iterations = len(new_bible_languages_url) * len(new_bible_books)
pbar = tqdm(total=total_iterations, desc="Overall Progress", unit="book")

try:
    for lang, root_url in new_bible_languages_url.items():
        bible_data_new[lang] = {}
        word_counts_new[lang] = 0

        # extract the parts of the URL
        try:
            parts = root_url.split("/")
            base_bible_url = f"https://www.bible.com/bible/{parts[4]}"
            version = root_url.split(".")[-1]
        except IndexError:
            print(f"Skipping invalid URL format for {lang}: {root_url}")
            pbar.update(len(new_bible_books))  
            continue  # skip to next language if error occurs

        for book in new_bible_books:
            pbar.set_description(f"Scraping {lang} - {book}")
            bible_data_new[lang][book] = {}

            ch = 1
            while True:
                url = f"{base_bible_url}/{book}.{ch}.{version}"
                driver.get(url)
                time.sleep(1)  # wait for page to load

                soup = BeautifulSoup(driver.page_source, "html.parser")
                
                # checks if end of chapters reached using "not available" marker
                not_available = soup.find("span", class_="ChapterContent_not-avaliable-span__WrOM_")
                if not_available:
                    break

                # if not yet end of chapters, extract
                chapter_content = soup.find_all("span", {"data-usfm": True})

                # mark as empty if no content found
                if not chapter_content:
                    bible_data_new[lang][book][ch] = ["MISSING"]
                else:
                    chapter_verses_new = []
                    chapter_word_count_new = 0

                    for verse in chapter_content:
                        # remove footnotes within the verse
                        for note in verse.find_all("span", class_=lambda x: x and x.startswith("ChapterContent_note")):
                            note.decompose()

                        # extract clean verse text
                        verse_text = verse.get_text(" ", strip=True)
                        if verse_text:
                            chapter_verses_new.append(verse_text)
                            verse_words_new = len(verse_text.split())
                            chapter_word_count_new += verse_words_new
                            total_verses_new += 1

                    # add verses
                    bible_data_new[lang][book][ch] = (
                        chapter_verses_new if chapter_verses_new else ["MISSING"]
                    )

                    # update word counts
                    word_counts_new[lang] += chapter_word_count_new
                    total_words_new += chapter_word_count_new
                    total_chapters_new += 1

                    pbar.set_postfix({
                        'Words': f"{total_words_new:,}",
                        'Verses': f"{total_verses_new:,}",
                        'Chapters': total_chapters_new
                    })

                ch += 1       # next chapter
                if ch > 100:  # safety cap
                    break

            pbar.update(1)

finally:
    driver.quit()
    pbar.close()
    print("Scraping complete.")

# summary statistics
print("\n" + "="*60)
if total_verses_new > 0 and total_chapters_new > 0:
    print(f"Total Words: {total_words_new:,}")
    print(f"Total Verses: {total_verses_new:,}")
    print(f"Total Chapters: {total_chapters_new}")
    print(f"Average Words per Verse: {total_words_new/total_verses_new:.1f}")
    print(f"Average Words per Chapter: {total_words_new/total_chapters_new:.1f}")
else:
    print("No data scraped or processed.")

print("\nWord Count by Language:")
print("-" * 30)
if total_words_new > 0:
    for lang_key, count in word_counts_new.items():
        if count > 0:
            percentage = (count / total_words_new) * 100
            print(f"{lang_key:12}: {count:8,} words ({percentage:.1f}%)")
        else:
            print(f"{lang_key:12}: {count:8,} words (0.0%) - Check availability/URL")
else:
    print("No words counted.")

Setting up WebDriver...
WebDriver ready.
WebDriver ready.


Scraping bikolano - REV: 100%|██████████| 162/162 [37:52<00:00, 14.03s/book, Words=1,140,710, Verses=48,662, Chapters=1464]

Scraping complete.

Total Words: 1,140,710
Total Verses: 48,662
Total Chapters: 1464
Average Words per Verse: 23.4
Average Words per Chapter: 779.2

Word Count by Language:
------------------------------
english     :  178,196 words (15.6%)
masbatenyo  :  197,039 words (17.3%)
tagalog     :  182,190 words (16.0%)
ilonggo     :  205,949 words (18.1%)
waray       :  198,936 words (17.4%)
bikolano    :  178,400 words (15.6%)


In [5]:
bible_data_new

{'english': {'MAT': {1: ['1 The book of the generation of Jesus Christ, the son of David, the son of Abraham.',
    '2 Abraham begat Isaac; and Isaac begat Jacob; and Jacob begat Judah and his brethren;',
    '3 and Judah begat Perez and Zerah of Tamar; and Perez begat Hezron; and Hezron begat Ram;',
    '4 and Ram begat Amminadab; and Amminadab begat Nahshon; and Nahshon begat Salmon;',
    '5 and Salmon begat Boaz of Rahab; and Boaz begat Obed of Ruth; and Obed begat Jesse;',
    '6 and Jesse begat David the king.',
    'And David begat Solomon of her that had been the wife of Uriah;',
    '7 and Solomon begat Rehoboam; and Rehoboam begat Abijah; and Abijah begat Asa;',
    '8 and Asa begat Jehoshaphat; and Jehoshaphat begat Joram; and Joram begat Uzziah;',
    '9 and Uzziah begat Jotham; and Jotham begat Ahaz; and Ahaz begat Hezekiah;',
    '10 and Hezekiah begat Manasseh; and Manasseh begat Amon; and Amon begat Josiah;',
    '11 and Josiah begat Jechoniah and his brethren, at the t

In [4]:
# Save bible data to text files
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

for lang, books in bible_data_new.items():
    output_file = output_dir / f"{lang}_raw.txt"
    
    with open(output_file, "w", encoding="utf-8") as f:
        for book, chapters in books.items():
            for chapter_num, verses in chapters.items():
                for verse_num, verse_text in enumerate(verses, start=1):
                    # Format: Book Chapter:Verse Text
                    f.write(f"{book} {chapter_num}:{verse_num} {verse_text}\n")
    
    print(f"Saved {lang} data to {output_file}")

print("\nAll files saved successfully!")

Saved english data to ..\data\raw\english_raw.txt
Saved masbatenyo data to ..\data\raw\masbatenyo_raw.txt
Saved tagalog data to ..\data\raw\tagalog_raw.txt
Saved ilonggo data to ..\data\raw\ilonggo_raw.txt
Saved waray data to ..\data\raw\waray_raw.txt
Saved bikolano data to ..\data\raw\bikolano_raw.txt

All files saved successfully!
Saved tagalog data to ..\data\raw\tagalog_raw.txt
Saved ilonggo data to ..\data\raw\ilonggo_raw.txt
Saved waray data to ..\data\raw\waray_raw.txt
Saved bikolano data to ..\data\raw\bikolano_raw.txt

All files saved successfully!
